In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

df = pd.read_csv("../data/comprovantes_pix_10000_anomalias.csv", sep=";")
df.drop(columns=['EndToEndId', 'Descricao'], inplace=True, errors='ignore')

# errors='coerce' transforma as datas impossíveis em "nulas" (NaT) em vez de quebrar o código
df['DataHora'] = pd.to_datetime(df['DataHora'], errors='coerce')

# Se a data ficou nula, marcamos como 1 (Data Inválida)
df['Data_Invalida'] = df['DataHora'].isna().astype(int)

# Preenchemos as datas quebradas com uma data genérica só para não dar erro de "Not a Number" na Célula 2
df['DataHora'] = df['DataHora'].fillna(pd.to_datetime('2000-01-01'))

print("Dados carregados com sucesso! Encontramos", df['Data_Invalida'].sum(), "datas impossíveis/anômalas.")

Dados carregados com sucesso! Encontramos 24 datas impossíveis/anômalas.


In [ ]:
# O .dt.hour e .dt.dayofweek roubam apenas a hora e o dia da semana (onde 0 é segunda e 6 é domingo).
# função lambda para dizer: Se o dia for 5 (sábado) ou 6 (domingo), marque 1 (É fim de semana). Senão, marque 0.
df['Hora'] = df['DataHora'].dt.hour
df['DiaDaSemana'] = df['DataHora'].dt.dayofweek
df['FimDeSemana'] = df['DiaDaSemana'].apply(lambda x: 1 if x >= 5 else 0)

# Horário comercial ganha 1 apenas se for entre 8h e 18h e se não for fim de semana. Já a madrugada ganha 1 se a hora for de 0 a 5.
df['Horario_Comercial'] = df.apply(lambda row: 1 if (8 <= row['Hora'] <= 18) and (row['FimDeSemana'] == 0) else 0, axis=1)
df['Madrugada'] = df['Hora'].apply(lambda x: 1 if 0 <= x <= 5 else 0)

# Isola os dias de pico bancário (salários e vales). 
df['Dia_do_Mes'] = df['DataHora'].dt.day
df['Dia_de_Pagamento'] = df['Dia_do_Mes'].apply(lambda x: 1 if x in [5, 6, 7, 20, 21, 22] else 0)

# Faz uma divisão por 1 (usando o %). Se o resto for zero, significa que não tem centavos (ex: R$ 500,00 redondo). 
df['Valor_Redondo'] = df['Valor'].apply(lambda x: 1 if x % 1 == 0 else 0)

df['Status_Pendente'] = df['Status'].apply(lambda x: 1 if x == 'Pendente' else 0)

# Compara se o banco de quem mandou o Pix é igual ao banco de quem recebeu. Fraudes frequentemente envolvem contas laranjas de bancos diferentes.
df['Mesmo_Banco'] = (df['Pagador_Banco'] == df['Recebedor_Banco']).astype(int)
# Conta quantos caracteres tem o documento. 
# Como o CPF tem até 14 caracteres (com os pontos) e o CNPJ tem 18, qualquer documento maior que 14 vira 1 (É Empresa/PJ).
df['Pagador_PJ'] = df['Pagador_CPF_CNPJ'].astype(str).apply(lambda x: 1 if len(x) > 14 else 0)
df['Recebedor_PJ'] = df['Recebedor_CPF_CNPJ'].astype(str).apply(lambda x: 1 if len(x) > 14 else 0)

# Aplica um logaritmo no dinheiro, porque na base pode ter um Pix de R$ 10 e outro de R$ 50.000. 
# Essa diferença muito grande distorce a análise, dessa forma o logaritmo "amassa" esses números para uma escala mais próxima, facilitando o cálculo da distância sem perder a proporção.
df['Valor_Log'] = np.log1p(df['Valor'])

# Normalização com o StandardScaler. 
# Assim, o "dinheiro" vai pesar na decisão do algoritmo exatamente o mesmo tanto que a variável "hora". 
features_para_escalar = ['Valor', 'Valor_Log', 'Hora']
scaler_std = StandardScaler()
df_std = pd.DataFrame(scaler_std.fit_transform(df[features_para_escalar]), columns=[f"{c}_std" for c in features_para_escalar])
df = pd.concat([df, df_std], axis=1)

print("Etapa 1 Concluída! Variáveis criadas e normalizadas.")

Etapa 1 Concluída! Variáveis criadas e normalizadas.
